In [ ]:
import os
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import torch
import sys

# --- Configuration ---
# Folder where the JSON files from the previous script are saved
INPUT_JSON_FOLDER = "/home/jupyter/Extracted_Text_v2"

# Folder to save the processed chunks with metadata and embeddings
# We'll save a new JSON file for each input JSON, containing a list of chunks
OUTPUT_CHUNKS_FOLDER = "/home/jupyter/Processed_Chunks_with_Embeddings"

# Chunking parameters
CHUNK_SIZE = 1000  # The maximum number of characters per chunk
CHUNK_OVERLAP = 200 # The number of characters to overlap between consecutive chunks

# Embedding model (choose a suitable model from sentence-transformers)
# 'all-MiniLM-L6-v2' is a good balance of speed and performance for many tasks
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'

# --- Setup ---
# Create the output folder for chunks
os.makedirs(OUTPUT_CHUNKS_FOLDER, exist_ok=True)
print(f"Input JSON folder: {INPUT_JSON_FOLDER}")
print(f"Output chunks folder ensured: {OUTPUT_CHUNKS_FOLDER}")
print(f"Chunk Size: {CHUNK_SIZE}, Chunk Overlap: {CHUNK_OVERLAP}")
print(f"Embedding Model: {EMBEDDING_MODEL_NAME}")

# Initialize the text splitter
# We use RecursiveCharacterTextSplitter which tries to split based on a list of characters
# (like newline, then space, etc.) to keep sentences/paragraphs together where possible.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len, # Use character length
    add_start_index=True # Keep track of where chunks start in the original text
)

# Initialize the embedding model
try:
    # Check for GPU availability
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
    print(f"Embedding model '{EMBEDDING_MODEL_NAME}' loaded successfully.")
except Exception as e:
    print(f"Error loading embedding model '{EMBEDDING_MODEL_NAME}': {e}", file=sys.stderr)
    print("Please ensure you have 'torch' and 'sentence-transformers' installed and that the model name is correct.", file=sys.stderr)
    sys.exit(1) # Exit if the model fails to load

# --- Functions ---

def split_text_into_chunks_with_metadata(text: str, metadata: dict) -> list[dict]:
    """
    Splits the input text into chunks and attaches metadata to each chunk.

    Args:
        text: The text content to split.
        metadata: A dictionary containing metadata for the entire document
                  (e.g., file_name, company_name, insurance_name).

    Returns:
        A list of dictionaries, where each dictionary represents a chunk
        and contains the chunk text and associated metadata.
    """
    if not text:
        print("Warning: Received empty text for chunking.")
        return []

    # Use the initialized text_splitter
    chunks = text_splitter.create_documents([text])

    processed_chunks = []
    for i, chunk in enumerate(chunks):
        chunk_metadata = metadata.copy() # Start with document metadata
        # Add chunk-specific metadata
        chunk_metadata.update({
            "chunk_index": i,
            "chunk_length": len(chunk.page_content),
            "start_index": chunk.metadata.get('start_index') # From text_splitter
        })

        processed_chunks.append({
            "text": chunk.page_content,
            "metadata": chunk_metadata
        })

    print(f"Split text into {len(processed_chunks)} chunks.")
    return processed_chunks

def generate_embedding(text):
    """
    Generates a single embedding for a text.

    Args:
        text: The text to embed.

    Returns:
        A list representing the embedding vector.
    """
    try:
        # Generate a single embedding for the text
        embedding = embedding_model.encode(text).tolist()
        return embedding
    except Exception as e:
        print(f"Error generating embedding: {e}", file=sys.stderr)
        return []  # Return empty list on error

def generate_embeddings_in_batches(chunks, batch_size=16):
    """
    Generates embeddings for a list of text chunks in batches.

    Args:
        chunks: A list of text chunks to embed.
        batch_size: The number of chunks to process in each batch.

    Returns:
        A list of embeddings for the input chunks.
    """
    # Handle single text case
    if isinstance(chunks, str):
        return generate_embedding(chunks)

    # Handle list of texts case
    embeddings = []
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        try:
            batch_embeddings = embedding_model.encode(batch, batch_size=batch_size).tolist()
            embeddings.extend(batch_embeddings)
        except Exception as e:
            print(f"Error generating embeddings for batch {i // batch_size + 1}: {e}", file=sys.stderr)
            embeddings.extend([[]] * len(batch))  # Add empty embeddings for failed chunks
    return embeddings

# --- Main Processing Logic ---
def process_all_json_files(input_folder: str, output_folder: str):
    """
    Reads JSON files from input_folder, chunks text, generates embeddings,
    and saves results to output_folder.
    """
    if not os.path.isdir(input_folder):
        print(f"Error: Input folder not found at {input_folder}", file=sys.stderr)
        return

    json_files = [f for f in os.listdir(input_folder) if f.endswith('.json')]
    print(f"Found {len(json_files)} JSON files in {input_folder}.")

    if not json_files:
        print("No JSON files found. Make sure the previous script ran successfully.")
        return

    for json_file in json_files:
        file_path = os.path.join(input_folder, json_file)
        print(f"\nProcessing file: {json_file}")

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            print(f"Error reading JSON file {json_file}: {e}", file=sys.stderr)
            continue # Skip to the next file

        # Extract data required for chunking and metadata
        extracted_text = data.get('extracted_text', '')
        # Use metadata already extracted
        document_metadata = {
            "file_name": data.get('file_name'),
            "company_name": data.get('company_name'),
            "insurance_name": data.get('insurance_name')
            # Add any other relevant document-level metadata here
        }

        if not extracted_text or len(extracted_text.strip()) < CHUNK_SIZE/2: # Simple check for minimal content
             print(f"Skipping '{json_file}': Extracted text is empty or too short.")
             continue

        # 1. Split text into chunks with metadata
        chunks_with_metadata = split_text_into_chunks_with_metadata(
            extracted_text,
            document_metadata
        )

        if not chunks_with_metadata:
            print(f"No chunks generated for '{json_file}'.")
            continue

        # 2. Generate embeddings for each chunk and prepare final output structure
        processed_chunks_output = []
        print(f"Generating embeddings for {len(chunks_with_metadata)} chunks...")
        for i, chunk_data in enumerate(chunks_with_metadata):
            chunk_text = chunk_data['text']
            chunk_metadata = chunk_data['metadata']

            # Generate a single embedding for the chunk text
            embedding = generate_embedding(chunk_text)

            if not embedding:
                 print(f"Skipping chunk {i} due to embedding error.")
                 continue # Skip this chunk if embedding failed

            # Verify embedding dimensions
            if i == 0:
                print(f"First embedding dimensions: {len(embedding)}")
                if len(embedding) != 384:
                    print(f"WARNING: Expected 384 dimensions but got {len(embedding)}!")

            # UPDATED: Flattened structure with individual metadata fields
            processed_chunks_output.append({
                "id": f"{chunk_metadata['file_name']}_{i}",  # Create a unique ID
                "embedding": embedding,
                "company_name": chunk_metadata.get('company_name', ''),
                "insurance_name": chunk_metadata.get('insurance_name', ''),
                "file_name": chunk_metadata.get('file_name', ''),
                "chunk_text": chunk_text,
                "chunk_index": chunk_metadata.get('chunk_index', i),
                "chunk_length": chunk_metadata.get('chunk_length', len(chunk_text)),
                "start_index": chunk_metadata.get('start_index', 0)
            })

            # Print progress every 10 chunks
            if (i + 1) % 10 == 0:
                print(f"  Processed {i + 1}/{len(chunks_with_metadata)} chunks...")

        # 3. Save the processed chunks and embeddings
        if processed_chunks_output:
            output_file_name = os.path.splitext(json_file)[0] + "_chunks.json"
            output_file_path = os.path.join(output_folder, output_file_name)

            try:
                with open(output_file_path, 'w', encoding='utf-8') as f:
                    json.dump(processed_chunks_output, f, indent=4, ensure_ascii=False)
                print(f"Successfully processed and saved {len(processed_chunks_output)} chunks to {output_file_name}")
            except Exception as e:
                print(f"Error saving processed chunks for {json_file} to {output_file_path}: {e}", file=sys.stderr)
        else:
            print(f"No valid chunks with embeddings generated for '{json_file}'.")


# --- Main Execution ---
if __name__ == "__main__":
    process_all_json_files(INPUT_JSON_FOLDER, OUTPUT_CHUNKS_FOLDER)
    print("\nEmbedding generation and chunk processing finished.")
    print(f"Processed chunks with embeddings saved to: {OUTPUT_CHUNKS_FOLDER}")